# 02 · Ground truth from the 2020 census

An audit needs a ground truth that covers every community, not only the places a survey happened to visit.
Mexico's 2020 census provides one: INEGI's locality-level results (ITER) report population, Indigenous
identity and dwelling conditions locality by locality, across the whole country. This notebook downloads
the census and Meta's Relative Wealth Index and builds one row per locality, with three definitions of
poverty, the Indigenous population share, and the map tile the locality falls in.

**Runtime:** about 15 minutes, mostly downloads. **Outputs:** the locality tables in `data/processed/`
(kept local, because the raw data carry their own licenses) and the aggregate build summary in
`results/real/tables/build_summary.json`.

In [ ]:
# Make the project importable; works in JupyterLab, VS Code and Colab without reinstalling.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from mapaudit import notebook as nb
from mapaudit import pipeline, report
from mapaudit.config import get_paths

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
MODE = nb.notebook_mode()  # "real" by default; CI sets MAPAUDIT_NOTEBOOK_MODE=demo (synthetic data)
print("Mode:", MODE)

## 1. Download the census file and the wealth map

In [ ]:
if MODE == "real":
    from mapaudit.config import load_config
    from mapaudit.download import download_all

    for message in download_all(load_config("real"), get_paths("real").raw):
        print(message)
else:
    print("Demo mode: the build step generates synthetic files instead.")

If an automatic download fails, the files can be fetched by hand and saved in `data/raw/` under the names
the messages give:

- **Census (ITER):** INEGI → Censo de Población y Vivienda 2020 → Datos abiertos → *Principales resultados
  por localidad (ITER)* → national CSV.
- **Relative Wealth Index:** the "Relative Wealth Index" dataset on the Humanitarian Data Exchange. If no
  Mexico file is available, `docs/data_card.md` describes the fallback, and the change is recorded in the
  deviations log of the pre-analysis plan.

## 2. Build the locality table

Aggregate rows are dropped and coordinates parsed. Each locality then gets its Indigenous population share,
dwelling-deprivation indicators, distance to the nearest city of 50,000 people, the three poverty
definitions, and the Relative Wealth Index tile that contains it.

In [ ]:
summary = pipeline.build(MODE)
pd.Series({k: v for k, v in summary.items() if k != "label_loadings"}, name="value", dtype=object).to_frame()

## 3. Sanity checks

Before using the table, I check it against known facts: the national count of localities and people,
whether the map's tile codes agree with its own coordinates, and whether each composite poverty index loads
positively on all of its components. Every row should read `pass`.

In [ ]:
nb.sanity_checks(summary, MODE)

## 4. A first look at the data

The first table counts the localities the analysis can use, those with suppressed dwelling data (INEGI
withholds detail for localities with fewer than three dwellings) and the people living in them, and the
Indigenous localities (at least 40% of the population in Indigenous households). The second lists the ten
states with the largest Indigenous population share.

In [ ]:
loc = pd.read_parquet(get_paths(MODE).localities)
pd.Series({
    "localities": len(loc),
    "rural, in scope": int(loc["in_scope"].sum()),
    "with dwelling data": int(loc["has_dwelling_data"].sum()),
    "without dwelling data (mostly < 3 dwellings)": int((~loc["has_dwelling_data"]).sum()),
    "people living in those": int(loc.loc[~loc["has_dwelling_data"], "POBTOT"].sum()),
    "Indigenous localities (>= 40%)": int((loc["indigenous"] == 1).sum()),
}, name="count").to_frame()

In [ ]:
by_state = (
    loc.assign(ind_pop=loc["POBTOT"] * loc["ind_share"])
    .groupby("NOM_ENT")[["POBTOT", "ind_pop"]]
    .sum()
)
by_state["ind_share"] = by_state["ind_pop"] / by_state["POBTOT"]
by_state.sort_values("ind_share", ascending=False).head(10).round(3)

How the two composite poverty definitions are built: their principal-component loadings. Every loading
should be positive, so that each component moves its index in the same direction.

In [ ]:
pd.DataFrame(summary["label_loadings"]).round(3)

## 5. Spot checks against INEGI's published tables

A few localities, pulled for comparison with INEGI's online tables. The output of this cell is cleared
before the notebook is published, because the data card commits to keeping locality-level rows off GitHub.

In [ ]:
PLACES = ["Chamula", "Tlaxiaco", "Valladolid"]  # localities to compare with INEGI's tables
cols = ["cvegeo", "NOM_ENT", "NOM_MUN", "NOM_LOC", "POBTOT", "ind_hh_share", "no_elec", "no_water", "dirt_floor"]
loc.loc[loc["NOM_LOC"].str.contains("|".join(PLACES), case=False, na=False), cols].head(15)